# P06 · tendon vibration — does the latency move?

**The question.** Everything so far has measured how *big* the response is. This measures **when
it happens**: the moment the EMG leaves its baseline, and the moment it peaks, in ms after the
pulse that evoked it.

Latency is the one measure that speaks to the **pathway** rather than the gain. A response that
grows or shrinks could be more or fewer fibres recruited; a response that arrives **later** has
taken a longer route — an extra synapse, or slower fibres. So:

> **Does tendon vibration change when the response arrives, and does it do so differently for
> cathodic and anodic stimulation?**

**Two numbers per muscle**, both averaged over the ten pulses:

| | |
|---|---|
| **onset** | the first moment the EMG leaves its own baseline — `K_SD` standard deviations of the pre-stimulus stretch, held for `MIN_RUN` samples. The conduction question, and the more fragile: a noisy baseline raises the bar the signal has to cross. |
| **peak** | the midpoint between the max and the min the peak-to-peak is made of. Far more stable, but it moves with the **shape** of the response as well as its timing. |
| **jitter** | how much the peak latency varies from pulse to pulse. A latency that wanders is not a latency. |

**Sampling here is ~0.8 ms**, so anything under about **1.5 ms** is not resolvable. Do not read a
1 ms difference.

**All eight forearm and hand muscles**, not the usual four — so every one needs a threshold, and
section 5 has one picker per control recording covering all of them at once.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from functions import (set_style, load_run, pretty, thresholds_for, threshold_source,
                       latency_table, fig_vibration, fig_group_summary,
                       threshold_picker, save_threshold_csv, load_threshold_csv, mt_file,
                       mt_stamp, warn_if_stale, fig_threshold_grid, fig_detection_grid)
set_style()
from functions.paper import motor_thresholds

## 1 · Config

In [ ]:
# ======== the recordings ========
D = "tSCS_CHUV_data/29-09-2026/"    # P06, 29-09-2026. Only the 20260929 files are this session.

ALL = {   # (protocol, polarity, block) -> csv.  "negative" in the log = cathodic.
 ("30 Hz",  "cathodic", "VIB off"): D + "Burst_autosave_20260929_163809_190ms.csv",       # 16:38
 ("30 Hz",  "cathodic", "VIB ON"):  D + "Burst_autosave_20260929_175220_421ms.csv",       # 17:52
 ("30 Hz",  "anodic",   "VIB off"): D + "Burst_autosave_20260929_163517_916ms.csv",       # 16:35
 ("30 Hz",  "anodic",   "VIB ON"):  D + "Burst_autosave_20260929_170919_038ms.csv",       # 17:09
 ("ARC-EX", "cathodic", "VIB off"): D + "Modulated_autosave_20260929_164440_099ms.csv",   # 16:44
 ("ARC-EX", "cathodic", "VIB ON"):  D + "Modulated_autosave_20260929_170027_460ms.csv",   # 17:00
 ("ARC-EX", "anodic",   "VIB off"): D + "Modulated_autosave_20260929_164216_867ms.csv",   # 16:42
 ("ARC-EX", "anodic",   "VIB ON"):  D + "Modulated_autosave_20260929_173301_845ms.csv",   # 17:33
}

PROTOCOLS  = ["30 Hz", "ARC-EX"]
POLARITIES = ["cathodic", "anodic"]
CONTROL    = "VIB off"
CONDITIONS = [CONTROL, "VIB ON"]
PANELS     = [f"{p} · {pol}" for p in PROTOCOLS for pol in POLARITIES]
RUN        = {(f"{p} · {pol}", b): ALL[(p, pol, b)]
              for p in PROTOCOLS for pol in POLARITIES for b in CONDITIONS}

VIBRATED = "Ext. digitorum (L)"      # the vibrator sat on the left wrist extensor

# ======== every forearm and hand muscle, not the usual four ========
MUSCLES = ["Thenar (L)", "Ext. digitorum (L)", "Flex. digitorum (L)", "Flex. carpi rad. (L)",
           "Thenar (R)", "Ext. digitorum (R)", "Flex. digitorum (R)", "Flex. carpi rad. (R)"]
# the proximal channels (deltoid, biceps, triceps, trapezius) are left out as everywhere else

# ======== the latency settings ========
K_SD     = 3.0    # how far from its own baseline the EMG must go to count as having started
MIN_RUN  = 2      # ...and for how many samples in a row
JIT_MAX  = 3.0    # a muscle whose peak latency jitters more than this (ms) is not a latency
# Sampling is ~0.8 ms, so anything under ~1.5 ms is not resolvable.
# ======================================

COND_COLOUR = {CONTROL: "#8C8C8C", "VIB ON": "#E67E22"}

KW = dict(n_pulses=10, resp_start_ms=8.0, resp_end_ms=None, guard_ms=1.0, min_snr=1.2,
          max_edge_frac=0.5, snr_on="median", anchor="mean", anchor_win_ms=3.0)

runs = RUN
for pan in PANELS:
    for b in CONDITIONS:
        f = runs[(pan, b)]
        amps = [m["amp_ma"] for m in load_run(f)[0]]
        print(f"{pan:20s} {b:10s} {f.split('/')[-1][-22:]}  {amps[0]:g}-{amps[-1]:g} mA"
              + ("   <- control" if b == CONTROL else ""))

## 2 · The motor thresholds you chose — all eight muscles

Each panel takes the threshold picked in its **`VIB off`** recording and reads both of its blocks
at it, muscle by muscle. Latency is measured at one intensity per muscle for the same reason
amplitude is: recruitment changes with current, and a response recruited from more fibres can peak
earlier even with no change in pathway.

`not picked` is expected at first — this notebook uses **eight** muscles where the others use four,
so the extra ones need picking. Section 5 has one picker per control recording and it covers all
eight at once.

In [ ]:
CHECK = {f"{pan} · {b}": runs[(pan, b)] for pan in PANELS for b in CONDITIONS}

picked = fig_threshold_grid(CHECK, MUSCLES, title="what is saved for each recording")

In [ ]:
MT = {pan: thresholds_for(runs[(pan, CONTROL)], MUSCLES, consecutive=2, **KW) for pan in PANELS}
for pan in PANELS:
    print(f"{pan:20s} {CONTROL:10s} {threshold_source(runs[(pan, CONTROL)])}")

USED = {f"{pan} · {b}": MT[pan] for pan in PANELS for b in CONDITIONS}
_ = fig_threshold_grid(CHECK, MUSCLES, thresholds=USED, share_gain=True,
                       title=f"the motor threshold used: each panel's {CONTROL}")

print(f"\n{'panel':20s}" + "".join(f"{m.split(' (')[0][:7] + m[-3:]:>13s}" for m in MUSCLES))
for pan in PANELS:
    print(f"{pan:20s}" + "".join(f"{str(MT[pan].get(m) or '-'):>13s}" for m in MUSCLES))

MT_STAMP = mt_stamp(runs)

## 3 · Where the latency is measured from

One panel per muscle at its own threshold, with the markers the latency is taken from. Check here
before reading section 4: a latency is only as good as the peak it is measured on, and a muscle
whose markers are scattered has no latency to report.

In [ ]:
TRY = dict(KW)
DET_MT = {f"{pan} · {b}": MT[pan] for pan in PANELS for b in CONDITIONS}
fig_detection_grid(CHECK, DET_MT, MUSCLES, edge_ms=1.0, jitter_ms=0.5,
                   title=f"peak detection at each panel's {CONTROL} threshold", **TRY)

## 4 · The latencies

### 4a · Onset and peak, in milliseconds

**Not a percentage** — these are absolute times in ms after the pulse, so the bars are read
against each other and against the dashed line at the control's value, not against 100.

Grey is `VIB off`, orange is `VIB ON`. The vibrated muscle is marked `*`.

A difference under **~1.5 ms** is below the sampling resolution and means nothing.

In [ ]:
warn_if_stale(runs, MT_STAMP, used=[(pan, CONTROL) for pan in PANELS])

lat = latency_table(runs, MT, MUSCLES, CONDITIONS, k_sd=K_SD, min_run=MIN_RUN,
                    match="common", **KW)

for key, lab in (("onset_ms", "onset — when the EMG leaves its baseline"),
                 ("peak_ms", "peak — the middle of the response")):
    fig_vibration(lat, PANELS, MUSCLES, CONDITIONS, key=key, colours=COND_COLOUR,
                  vibrated={pan: VIBRATED for pan in PANELS},
                  ylabel="ms after the pulse", ref=None, ylim=(0, 30),
                  titles={pan: pan for pan in PANELS},
                  title=f"P06 · 29-09 — {lab}", save=None)

### The numbers, with the jitter that says whether to trust them

In [ ]:
print(f"{'panel':20s}{'muscle':22s}{'mA':>5s}"
      + "".join(f"{b[:7] + ' onset':>15s}{b[:7] + ' peak':>14s}{b[:7] + ' jit':>13s}"
                for b in CONDITIONS))
for pan in PANELS:
    for m in MUSCLES:
        if not any((pan, b, m) in lat for b in CONDITIONS):
            continue
        row = ""
        for b in CONDITIONS:
            v = lat.get((pan, b, m), {})
            o, p_, j = v.get("onset_ms"), v.get("peak_ms"), v.get("jitter_ms")
            flag = " !" if (j is not None and j == j and j > JIT_MAX) else "  "
            row += (f"{(f'{o:.1f}' if o == o else '-'):>15s}"
                    f"{(f'{p_:.1f}' if p_ == p_ else '-'):>14s}"
                    f"{(f'{j:.1f}' if j == j else '-') + flag:>13s}")
        amp = next((lat[(pan, b, m)]["amp"] for b in CONDITIONS if (pan, b, m) in lat), None)
        star = " *" if m == VIBRATED else "  "
        print(f"{pan:20s}{m + star:22s}{amp:5g}{row}")
print(f"\nonset/peak in ms after the pulse; jit = SD of the peak latency over the 10 pulses")
print(f"!  = jitter over {JIT_MAX:g} ms, so that latency is not stable enough to compare")

### 4b · The change, in milliseconds

The same thing as a **difference**: `VIB ON` minus `VIB off`, so a positive bar means the response
arrived **later** with the vibrator running. Zero is no change, and the band of ±1.5 ms is the
sampling resolution — a bar inside it is not a result.

In [ ]:
delta = {}
for pan in PANELS:
    for m in MUSCLES:
        a, b = lat.get((pan, CONTROL, m)), lat.get((pan, "VIB ON", m))
        if not a or not b:
            continue
        delta[(pan, "VIB ON", m)] = dict(
            onset_ms=b.get("onset_ms", np.nan) - a.get("onset_ms", np.nan),
            peak_ms=b.get("peak_ms", np.nan) - a.get("peak_ms", np.nan),
            snr=min(a.get("snr", np.nan), b.get("snr", np.nan)))

for key, lab in (("onset_ms", "onset"), ("peak_ms", "peak")):
    fig_vibration(delta, PANELS, MUSCLES, ["VIB ON"], key=key,
                  colours={"VIB ON": "#E67E22"},
                  vibrated={pan: VIBRATED for pan in PANELS},
                  ylabel="ms  (VIB ON − VIB off)", ref=0, ylim=(-8, 8),
                  titles={pan: pan for pan in PANELS},
                  title=f"P06 · 29-09 — change in {lab} latency with vibration", save=None)

### 4c · Pooled — does polarity behave differently?

The latency change averaged over the muscles, split by polarity, with every muscle as a dot. This
is the comparison the notebook exists for: if vibration delays one polarity and not the other,
they are not travelling the same route.

In [ ]:
DOT = {f"{p} · {pol}": c for (p, pol), c in
       zip([(p, pol) for p in PROTOCOLS for pol in POLARITIES],
           ["#4D4D4D", "#9A9A9A", "#8C2317", "#C0392B"])}

for key, lab in (("onset_ms", "onset"), ("peak_ms", "peak")):
    fig_group_summary(delta, PANELS, MUSCLES, ["VIB ON"], key=key,
                      colours={"VIB ON": "#B0B0B0"}, dot_colours=DOT, err="sd",
                      ylabel="ms  (VIB ON − VIB off)", ref=0, ylim=(-8, 8),
                      title=f"change in {lab} latency — every panel, muscle by muscle", save=None)

## 5 · Choose the motor thresholds

**Four cells**, one per panel — the `VIB off` recording, which sets the intensity for both of that
panel's blocks. Each picker shows **all eight muscles**, so one pass per cell covers them.

Move the slider, press **Set to slider**, then **Save**, and re-run from section 2.

In [ ]:
def repick(key):
    """Open the picker for one recording, with a Save button. All eight muscles at once."""
    csv = runs[key]
    meta_, t_, sig_ = load_run(csv)
    chans = [c for c in sig_ if c != "Trigger A" and pretty(c) in MUSCLES]
    prev = load_threshold_csv(mt_file(csv), by="channel") if os.path.exists(mt_file(csv)) else None
    detected = motor_thresholds([dict(label="_", csv=csv)], [pretty(c) for c in chans],
                                consecutive=2, verbose=False, **KW)["_"]
    print(f"{key}  {csv.split('/')[-1]}  |  {[m['amp_ma'] for m in meta_]} mA"
          + ("  |  your picks reloaded" if prev else "  |  nothing picked yet"))
    return threshold_picker(meta_, t_, sig_, chans, xlim=(-20, 130), picks=prev, key=csv,
                            suggest=detected, suggest_label="detected",
                            on_save=lambda p_: save_threshold_csv(p_, chans, csv, meta=meta_))

In [ ]:
repick(('30 Hz · cathodic', 'VIB off'))

In [ ]:
repick(('30 Hz · anodic', 'VIB off'))

In [ ]:
repick(('ARC-EX · cathodic', 'VIB off'))

In [ ]:
repick(('ARC-EX · anodic', 'VIB off'))